# Chapter 31: Camera Models

<a href="../lite/lab/index.html?path=ch31_camera_models.ipynb" target="_blank" style="display:inline-block;padding:8px 16px;background:#1976d2;color:white;border-radius:4px;text-decoration:none;font-weight:bold">▶ Open in JupyterLite (editable, no install)</a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Line3DCollection

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

Hold your phone camera up. Every pixel on the screen is a ray shooting out from the lens
into the 3D world. The **camera model** is the mathematical description of which 3D points
land on which pixels. Get it right, and you can measure real world distances from photos.

```{admonition} What you will build
:class: tip

- Project 3D objects (cubes, grids, landmarks) to 2D images using the pinhole camera model
- See how focal length changes the field of view (wide angle vs telephoto)
- Calibrate a camera from known 3D to 2D correspondences using DLT
- Visualize the effect of lens distortion on straight lines

**Real world application:** Camera calibration is the first step in any visual SLAM pipeline. After this chapter, you will understand the math that turns 3D scenes into 2D images and how to estimate camera parameters.
```

```{admonition} Libraries and tools used in practice
:class: note

In this chapter we implement everything from scratch for learning. In production, engineers use these libraries:

| Library / Tool | What it does |
|---|---|
| **OpenCV calibrateCamera** | Industry standard camera calibration using Zhang's method with checkerboards |
| **OpenCV undistort** | Lens distortion correction |
| **Kalibr** | Multi-camera and camera-IMU calibration toolbox |

Implementing from scratch teaches you **why** these tools work. Using them in production saves you from reinventing the wheel.
```

```{admonition} Libraries and tools used in practice
:class: note

In this chapter we implement everything from scratch for learning. In production, engineers use these libraries:

| Library / Tool | What it does |
|---|---|
| **OpenCV calibrateCamera** | Industry standard camera calibration using Zhang's method with checkerboards |
| **OpenCV undistort** | Lens distortion correction |
| **Kalibr** | Multi-camera and camera-IMU calibration toolbox |

Implementing from scratch teaches you **why** these tools work. Using them in production saves you from reinventing the wheel.
```

## 31.1 Projection

The **pinhole camera model** projects a 3D point $\mathbf{P} = [X, Y, Z]^T$ to a 2D pixel:

$$\begin{bmatrix} u \\ v \\ 1 \end{bmatrix} = \frac{1}{Z} K \begin{bmatrix} X \\ Y \\ Z \end{bmatrix}$$

where $K$ is the **intrinsic matrix**:

$$K = \begin{bmatrix} f_x & 0 & c_x \\ 0 & f_y & c_y \\ 0 & 0 & 1 \end{bmatrix}$$

The projection discards the depth $Z$; this is why a single image cannot recover 3D structure on its own.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
fx, fy = 500, 500           # focal length (pixels)
cx, cy = 320, 240           # principal point (image centre)
img_w, img_h = 640, 480     # image size
cube_size = 1.0             # side length of the cube (m)
cube_center = np.array([0.0, 0.0, 5.0])  # cube centre in camera frame
n_landmarks = 12            # random landmarks
# ─────────────────────────────────────────────────────────────────────────────

K = np.array([[fx, 0, cx],
              [0, fy, cy],
              [0,  0,  1]], dtype=float)

# Cube vertices
offsets = np.array([[dx, dy, dz]
                    for dx in [-1, 1]
                    for dy in [-1, 1]
                    for dz in [-1, 1]]) * cube_size / 2
cube_pts = cube_center + offsets

# Ground plane grid (Z from 3 to 8, X from -3 to 3)
gx = np.linspace(-3, 3, 7)
gz = np.linspace(3, 8, 6)
ground_lines = []
for z in gz:
    ground_lines.append(np.column_stack([gx, np.full_like(gx, 1.5), np.full_like(gx, z)]))
for x in gx:
    ground_lines.append(np.column_stack([np.full_like(gz, x), np.full_like(gz, 1.5), gz]))

# Random landmarks
np.random.seed(7)
landmarks = np.column_stack([
    np.random.uniform(-2, 2, n_landmarks),
    np.random.uniform(-1.5, 1.0, n_landmarks),
    np.random.uniform(3, 7, n_landmarks)
])

# Projection helper
def project_points(K, pts_3d):
    proj = (K @ pts_3d.T).T
    pix = proj[:, :2] / proj[:, 2:3]
    return pix

cube_pix = project_points(K, cube_pts)
lm_pix   = project_points(K, landmarks)

edges = [(0,1),(0,2),(0,4),(1,3),(1,5),(2,3),(2,6),(3,7),(4,5),(4,6),(5,7),(6,7)]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Top view (X vs Z)
ax = axes[0]
ax.set_title('3D Scene (top view, X vs Z)', fontsize=13)
ax.scatter(cube_pts[:, 0], cube_pts[:, 2], c='steelblue', s=50, zorder=5)
for i, j in edges:
    ax.plot([cube_pts[i,0], cube_pts[j,0]],
            [cube_pts[i,2], cube_pts[j,2]], 'steelblue', lw=1)
ax.scatter(landmarks[:, 0], landmarks[:, 2], c='orange', s=30, marker='^', label='landmarks')
ax.plot(0, 0, 'ko', ms=10, label='camera')
ax.set_xlabel('X (m)'); ax.set_ylabel('Z (m)'); ax.legend(fontsize=9)

# Side view (Z vs Y)
ax = axes[1]
ax.set_title('3D Scene (side view, Z vs Y)', fontsize=13)
ax.scatter(cube_pts[:, 2], cube_pts[:, 1], c='steelblue', s=50, zorder=5)
for i, j in edges:
    ax.plot([cube_pts[i,2], cube_pts[j,2]],
            [cube_pts[i,1], cube_pts[j,1]], 'steelblue', lw=1)
ax.scatter(landmarks[:, 2], landmarks[:, 1], c='orange', s=30, marker='^')
ax.plot(0, 0, 'ko', ms=10)
ax.set_xlabel('Z (m)'); ax.set_ylabel('Y (m)')

# Projected image
ax = axes[2]
ax.set_title('Projected image (pinhole)', fontsize=13)
ax.scatter(cube_pix[:, 0], cube_pix[:, 1], c='tomato', s=50, zorder=5, label='cube')
for i, j in edges:
    ax.plot([cube_pix[i,0], cube_pix[j,0]],
            [cube_pix[i,1], cube_pix[j,1]], 'tomato', lw=1.5)
ax.scatter(lm_pix[:, 0], lm_pix[:, 1], c='orange', s=30, marker='^', label='landmarks')
# Ground grid
for line in ground_lines:
    gpix = project_points(K, line)
    inside = (gpix[:,0] >= 0) & (gpix[:,0] < img_w) & (gpix[:,1] >= 0) & (gpix[:,1] < img_h)
    if inside.any():
        ax.plot(gpix[inside, 0], gpix[inside, 1], color='forestgreen', lw=0.5, alpha=0.5)
ax.set_xlim(0, img_w); ax.set_ylim(img_h, 0)
ax.set_xlabel('u (pixels)'); ax.set_ylabel('v (pixels)')
ax.set_aspect('equal'); ax.legend(fontsize=9)

plt.tight_layout()
plt.show()
print(f'Intrinsic matrix K:\n{K}')

**Key observations:**
- Points farther from the optical axis project toward the image edges.
- The ground grid lines converge to a **vanishing point** at the principal point; this is a hallmark of perspective projection.
- Changing `cube_center[2]` (depth) scales the projected size: double the depth, half the size.

## 31.2 Intrinsics: Focal Length and Field of View

The **focal length** controls the angular field of view (FOV):

$$\text{FOV}_x = 2 \arctan\!\left(\frac{w}{2 f_x}\right)$$

A short focal length yields a **wide angle** view (more of the scene is visible but with more distortion at the edges).
A long focal length yields a **telephoto** view (zoomed in, narrow FOV).

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
focal_lengths = [200, 500, 1000]   # wide, normal, telephoto
labels        = ['Wide (f=200)', 'Normal (f=500)', 'Telephoto (f=1000)']
# ─────────────────────────────────────────────────────────────────────────────

# Build a richer scene: 3D points forming a house shape
house_pts = np.array([
    [-1, -1, 5], [ 1, -1, 5], [ 1, 1, 5], [-1, 1, 5],   # front face
    [-1, -1, 7], [ 1, -1, 7], [ 1, 1, 7], [-1, 1, 7],   # back face
    [ 0, -1.8, 6],                                        # roof peak
    [-2,  1, 4], [ 2,  1, 4], [-2,  1, 8], [ 2,  1, 8],  # ground corners
], dtype=float)

house_edges = [(0,1),(1,2),(2,3),(3,0),  # front
               (4,5),(5,6),(6,7),(7,4),  # back
               (0,4),(1,5),(2,6),(3,7),  # sides
               (0,8),(1,8),(4,8),(5,8)]  # roof

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, f, lab in zip(axes, focal_lengths, labels):
    Kf = np.array([[f, 0, cx], [0, f, cy], [0, 0, 1]], dtype=float)
    pix = project_points(Kf, house_pts)
    fov = 2 * np.degrees(np.arctan(img_w / (2 * f)))
    ax.set_title(f'{lab}\nFOV = {fov:.1f} deg', fontsize=12)
    ax.scatter(pix[:, 0], pix[:, 1], c='steelblue', s=40, zorder=5)
    for i, j in house_edges:
        ax.plot([pix[i,0], pix[j,0]], [pix[i,1], pix[j,1]], 'steelblue', lw=1.2)
    ax.set_xlim(0, img_w); ax.set_ylim(img_h, 0)
    ax.set_xlabel('u (pixels)'); ax.set_ylabel('v (pixels)')
    ax.set_aspect('equal')

plt.suptitle('Same 3D scene, three focal lengths', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**Key observations:**
- Doubling the focal length doubles the projected size of every object (the image "zooms in").
- Wide angle lenses capture more context but exaggerate perspective.
- For SLAM, a moderate FOV (40 to 70 degrees) gives a good balance between coverage and feature stability.

## 31.3 Extrinsics: Camera Pose in the World

The **extrinsic matrix** $[R \mid t]$ converts world coordinates to camera coordinates:

$$\mathbf{P}_{\text{cam}} = R\,(\mathbf{P}_{\text{world}} - \mathbf{t}_{\text{world}})$$

Combining intrinsics and extrinsics gives the full projection:

$$\mathbf{p} = K\,[R \mid -R\,\mathbf{t}]\, \begin{bmatrix} \mathbf{P}_{\text{world}} \\ 1 \end{bmatrix}$$

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
world_points = house_pts.copy()
# Four camera poses: (position, yaw_deg, pitch_deg)
cam_poses = [
    {'pos': np.array([0, 0, 0]),    'yaw': 0,   'pitch': 0,   'label': 'Front view'},
    {'pos': np.array([-4, 0, 6]),   'yaw': 90,  'pitch': 0,   'label': 'Left view'},
    {'pos': np.array([4, 0, 6]),    'yaw': -90, 'pitch': 0,   'label': 'Right view'},
    {'pos': np.array([0, -5, 6]),   'yaw': 0,   'pitch': 60,  'label': 'Top view'},
]
# ─────────────────────────────────────────────────────────────────────────────

def rotation_matrix(yaw_deg, pitch_deg):
    """Build R = Rx(pitch) @ Ry(yaw)."""
    y = np.radians(yaw_deg)
    p = np.radians(pitch_deg)
    Ry = np.array([[ np.cos(y), 0, np.sin(y)],
                   [ 0,         1, 0         ],
                   [-np.sin(y), 0, np.cos(y)]])
    Rx = np.array([[1, 0,          0         ],
                   [0, np.cos(p), -np.sin(p) ],
                   [0, np.sin(p),  np.cos(p) ]])
    return Rx @ Ry

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, cp in zip(axes, cam_poses):
    R = rotation_matrix(cp['yaw'], cp['pitch'])
    t = cp['pos']
    pts_cam = (R @ (world_points - t).T).T
    visible = pts_cam[:, 2] > 0.1
    if visible.sum() == 0:
        ax.set_title(cp['label'] + ' (nothing visible)'); continue
    pix = project_points(K, pts_cam[visible])
    inside = (pix[:,0]>=0) & (pix[:,0]<img_w) & (pix[:,1]>=0) & (pix[:,1]<img_h)
    ax.scatter(pix[inside, 0], pix[inside, 1], c='steelblue', s=50, zorder=5)
    # Draw edges where both endpoints visible and inside
    vis_idx = np.where(visible)[0]
    idx_map = {orig: new for new, orig in enumerate(vis_idx)}
    for i, j in house_edges:
        if i in idx_map and j in idx_map:
            ii, jj = idx_map[i], idx_map[j]
            if inside[ii] and inside[jj]:
                ax.plot([pix[ii,0], pix[jj,0]], [pix[ii,1], pix[jj,1]],
                        'steelblue', lw=1.2)
    ax.set_xlim(0, img_w); ax.set_ylim(img_h, 0)
    ax.set_xlabel('u'); ax.set_ylabel('v')
    ax.set_title(f"{cp['label']}\nyaw={cp['yaw']}  pitch={cp['pitch']}", fontsize=11)
    ax.set_aspect('equal')

plt.suptitle('Same scene from four camera poses', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### Extrinsic matrix structure

The $3 \times 4$ projection matrix $P = K [R \mid -Rt]$ contains six extrinsic parameters (three for rotation, three for translation). Let us print $P$ for each viewpoint above.

In [ ]:
for cp in cam_poses:
    R = rotation_matrix(cp['yaw'], cp['pitch'])
    t = cp['pos'].reshape(3, 1)
    Rt = np.hstack([R, -R @ t])  # 3x4
    P = K @ Rt                    # 3x4 projection matrix
    print(f"\n{cp['label']} (yaw={cp['yaw']}, pitch={cp['pitch']})")
    print(f"R =\n{R.round(3)}")
    print(f"t = {cp['pos']}")
    print(f"P =\n{P.round(1)}")

## 31.4 Calibration via DLT

**Camera calibration** estimates the intrinsic matrix $K$ from known 3D to 2D correspondences.
The **Direct Linear Transform (DLT)** estimates the $3 \times 4$ projection matrix $P$ first,
then extracts $K$, $R$, $t$ via RQ decomposition.

Given $N$ correspondences $(\mathbf{P}_i, \mathbf{p}_i)$ we stack the constraint
$\mathbf{p}_i \times (P \mathbf{P}_i^h) = 0$ into a homogeneous system $A \mathbf{p}_{\text{vec}} = 0$
and solve with SVD.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
np.random.seed(42)
grid_rows, grid_cols = 7, 9          # checkerboard inner corners
square_size = 0.03                   # 30 mm squares
pixel_noise_sigma = 0.8              # pixel noise std (try 0, 0.5, 2.0)
true_fx, true_fy = 520, 520
true_cx, true_cy = 325, 240
# ─────────────────────────────────────────────────────────────────────────────

K_true = np.array([[true_fx, 0, true_cx],
                   [0, true_fy, true_cy],
                   [0,  0,  1]], dtype=float)

# Checkerboard corners in 3D (Z=0 plane)
pts3d = np.array([[c * square_size, r * square_size, 0.0]
                  for r in range(grid_rows)
                  for c in range(grid_cols)])

# Camera looking at the board: translate and rotate slightly
R_cal = rotation_matrix(5, 10)
t_cal = np.array([0.1, 0.08, 0.5])
pts_cam = (R_cal @ (pts3d - t_cal).T).T
pix_true = project_points(K_true, pts_cam)

# Add noise
pix_noisy = pix_true + np.random.normal(0, pixel_noise_sigma, pix_true.shape)

# ---- DLT: estimate P from correspondences ----
N = len(pts3d)
A = np.zeros((2 * N, 12))
for i in range(N):
    X, Y, Z = pts3d[i]
    u, v = pix_noisy[i]
    A[2*i]   = [X, Y, Z, 1,  0, 0, 0, 0,  -u*X, -u*Y, -u*Z, -u]
    A[2*i+1] = [0, 0, 0, 0,  X, Y, Z, 1,  -v*X, -v*Y, -v*Z, -v]

_, _, Vt = np.linalg.svd(A)
P_est = Vt[-1].reshape(3, 4)

# ---- Extract K, R, t via RQ decomposition ----
M = P_est[:, :3]
# RQ decomposition: flip, do QR, flip back
from scipy.linalg import qr
M_flip = np.flipud(M).T
Q_flip, R_flip = qr(M_flip)
R_upper = np.flipud(R_flip.T)
R_upper = R_upper[:, ::-1]
Q_orth  = Q_flip.T[::-1]

# Force positive diagonal in K
D = np.diag(np.sign(np.diag(R_upper)))
K_est = R_upper @ D
R_est = D @ Q_orth

# Normalise so K[2,2] = 1
K_est = K_est / K_est[2, 2]

print('True K:')
print(K_true)
print('\nEstimated K (DLT + RQ):')
print(K_est.round(2))
print(f'\nfx error: {abs(K_est[0,0] - true_fx):.2f} px')
print(f'fy error: {abs(K_est[1,1] - true_fy):.2f} px')
print(f'cx error: {abs(K_est[0,2] - true_cx):.2f} px')
print(f'cy error: {abs(K_est[1,2] - true_cy):.2f} px')

In [ ]:
# Visualise the calibration grid: true vs noisy projections
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.scatter(pix_true[:, 0], pix_true[:, 1], c='steelblue', s=20, label='true')
ax.scatter(pix_noisy[:, 0], pix_noisy[:, 1], c='tomato', s=12, marker='x', label='noisy')
ax.set_xlim(0, img_w); ax.set_ylim(img_h, 0)
ax.set_title('Calibration grid projections', fontsize=13)
ax.set_xlabel('u'); ax.set_ylabel('v')
ax.legend(); ax.set_aspect('equal')

# Reprojection using estimated P
pts3d_h = np.hstack([pts3d, np.ones((N, 1))])
reproj_h = (P_est @ pts3d_h.T).T
reproj = reproj_h[:, :2] / reproj_h[:, 2:3]

ax = axes[1]
ax.scatter(pix_noisy[:, 0], pix_noisy[:, 1], c='tomato', s=20, label='measured')
ax.scatter(reproj[:, 0], reproj[:, 1], c='forestgreen', s=12, marker='+', label='reprojected')
ax.set_xlim(0, img_w); ax.set_ylim(img_h, 0)
reproj_err = np.linalg.norm(reproj - pix_noisy, axis=1).mean()
ax.set_title(f'Reprojection (mean error = {reproj_err:.2f} px)', fontsize=13)
ax.set_xlabel('u'); ax.set_ylabel('v')
ax.legend(); ax.set_aspect('equal')

plt.tight_layout()
plt.show()

## 31.5 Distortion

Real lenses are not perfect pinholes. **Radial distortion** bends straight lines:

$$\begin{aligned}
x_d &= x\,(1 + k_1 r^2 + k_2 r^4) \\
y_d &= y\,(1 + k_1 r^2 + k_2 r^4)
\end{aligned}$$

where $(x, y)$ are normalised (undistorted) coordinates and $r^2 = x^2 + y^2$.

- $k_1 < 0$: **barrel** distortion (lines bow outward)
- $k_1 > 0$: **pincushion** distortion (lines bow inward)

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
k1_barrel     = -0.3      # barrel distortion (try -0.1, -0.5)
k2_barrel     =  0.05
k1_pincushion =  0.3      # pincushion distortion
k2_pincushion = -0.05
grid_n = 15               # grid density
# ─────────────────────────────────────────────────────────────────────────────

def apply_distortion(x, y, k1, k2):
    r2 = x**2 + y**2
    factor = 1 + k1 * r2 + k2 * r2**2
    return x * factor, y * factor

# Create a normalised grid
u_lin = np.linspace(-1, 1, grid_n)
v_lin = np.linspace(-0.75, 0.75, grid_n)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
titles = ['No distortion', f'Barrel (k1={k1_barrel})', f'Pincushion (k1={k1_pincushion})']
params = [(0, 0), (k1_barrel, k2_barrel), (k1_pincushion, k2_pincushion)]

for ax, (k1, k2), title in zip(axes, params, titles):
    ax.set_title(title, fontsize=12)
    # horizontal lines
    for v0 in v_lin:
        xs = u_lin.copy()
        ys = np.full_like(xs, v0)
        xd, yd = apply_distortion(xs, ys, k1, k2)
        ax.plot(xd, yd, 'steelblue', lw=0.8)
    # vertical lines
    for u0 in u_lin:
        ys = v_lin.copy()
        xs = np.full_like(ys, u0)
        xd, yd = apply_distortion(xs, ys, k1, k2)
        ax.plot(xd, yd, 'steelblue', lw=0.8)
    ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.2, 1.2)
    ax.set_aspect('equal')
    ax.set_xlabel('x (normalised)'); ax.set_ylabel('y (normalised)')

plt.suptitle('Radial distortion effects on a regular grid', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### Distortion magnitude map

The displacement caused by distortion grows rapidly with distance from the optical centre. The plot below shows the magnitude of the distortion vector $\| \mathbf{p}_d - \mathbf{p} \|$ across the image.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
k1_map, k2_map = -0.3, 0.05
# ─────────────────────────────────────────────────────────────────────────────

xg, yg = np.meshgrid(np.linspace(-1, 1, 200), np.linspace(-0.75, 0.75, 150))
r2 = xg**2 + yg**2
factor = 1 + k1_map * r2 + k2_map * r2**2
dx = xg * factor - xg
dy = yg * factor - yg
mag = np.sqrt(dx**2 + dy**2)

fig, ax = plt.subplots(figsize=(9, 6))
im = ax.imshow(mag, extent=[-1, 1, -0.75, 0.75], origin='lower', cmap='inferno')
plt.colorbar(im, ax=ax, label='displacement (normalised)')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title(f'Distortion magnitude (k1={k1_map}, k2={k2_map})', fontsize=13)
plt.tight_layout()
plt.show()

## Capstone: Complete Camera Pipeline

We now run the entire pipeline end to end:

1. Define a set of 3D points and a camera with known $K$ and pose.
2. Project the points to pixels.
3. Add realistic pixel noise.
4. Use the noisy 2D to 3D correspondences to estimate $K$ via DLT.
5. Project **new** 3D points with the estimated $K$ and compare to the true projection.
6. Report calibration error and reprojection error.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
np.random.seed(99)
n_calib_pts  = 80       # calibration points
n_test_pts   = 20       # new test points
noise_sigma  = 1.0      # pixel noise (try 0.5, 1.0, 3.0)
true_f       = 510.0
true_c       = np.array([320.0, 240.0])
# ─────────────────────────────────────────────────────────────────────────────

K_gt = np.array([[true_f, 0, true_c[0]],
                 [0, true_f, true_c[1]],
                 [0, 0, 1]], dtype=float)

# Camera pose
R_cap = rotation_matrix(8, 5)
t_cap = np.array([0.2, -0.1, 0.0])

# 3D calibration points on a plane Z=0
calib_3d = np.column_stack([
    np.random.uniform(-0.15, 0.15, n_calib_pts),
    np.random.uniform(-0.10, 0.10, n_calib_pts),
    np.zeros(n_calib_pts)
])

# Project and add noise
pts_cam_cap = (R_cap @ (calib_3d - t_cap).T).T
pix_true_cap = project_points(K_gt, pts_cam_cap)
pix_noisy_cap = pix_true_cap + np.random.normal(0, noise_sigma, pix_true_cap.shape)

# DLT estimation
A_cap = np.zeros((2 * n_calib_pts, 12))
for i in range(n_calib_pts):
    X, Y, Z = calib_3d[i]
    u, v = pix_noisy_cap[i]
    A_cap[2*i]   = [X, Y, Z, 1, 0, 0, 0, 0, -u*X, -u*Y, -u*Z, -u]
    A_cap[2*i+1] = [0, 0, 0, 0, X, Y, Z, 1, -v*X, -v*Y, -v*Z, -v]

_, _, Vt_cap = np.linalg.svd(A_cap)
P_cap = Vt_cap[-1].reshape(3, 4)

# Extract K
M_cap = P_cap[:, :3]
M_flip = np.flipud(M_cap).T
Q_f, R_f = qr(M_flip)
R_up = np.flipud(R_f.T)
R_up = R_up[:, ::-1]
D2 = np.diag(np.sign(np.diag(R_up)))
K_cap_est = R_up @ D2
K_cap_est = K_cap_est / K_cap_est[2, 2]

# Test on NEW points
test_3d = np.column_stack([
    np.random.uniform(-0.12, 0.12, n_test_pts),
    np.random.uniform(-0.08, 0.08, n_test_pts),
    np.zeros(n_test_pts)
])
test_cam = (R_cap @ (test_3d - t_cap).T).T
pix_test_true = project_points(K_gt, test_cam)

test_h = np.hstack([test_3d, np.ones((n_test_pts, 1))])
reproj_test_h = (P_cap @ test_h.T).T
pix_test_est = reproj_test_h[:, :2] / reproj_test_h[:, 2:3]

test_err = np.linalg.norm(pix_test_est - pix_test_true, axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.scatter(pix_noisy_cap[:, 0], pix_noisy_cap[:, 1], c='steelblue', s=15, alpha=0.5, label='calib (noisy)')
ax.scatter(pix_test_true[:, 0], pix_test_true[:, 1], c='forestgreen', s=40, marker='o', label='test (true)')
ax.scatter(pix_test_est[:, 0], pix_test_est[:, 1], c='tomato', s=25, marker='x', label='test (estimated K)')
ax.set_xlim(0, img_w); ax.set_ylim(img_h, 0)
ax.set_title('Capstone: calibration + test reprojection', fontsize=13)
ax.legend(fontsize=9); ax.set_aspect('equal')
ax.set_xlabel('u'); ax.set_ylabel('v')

ax = axes[1]
ax.bar(range(n_test_pts), test_err, color='orange', edgecolor='k', lw=0.3)
ax.axhline(test_err.mean(), color='tomato', ls='--', label=f'mean = {test_err.mean():.2f} px')
ax.set_xlabel('test point index'); ax.set_ylabel('reprojection error (px)')
ax.set_title('Per point reprojection error on test set', fontsize=13)
ax.legend()

plt.tight_layout()
plt.show()

print('=== Calibration Report ===')
print(f'True K:\n{K_gt}')
print(f'\nEstimated K:\n{K_cap_est.round(2)}')
print(f'\nfx error: {abs(K_cap_est[0,0] - true_f):.2f} px')
print(f'fy error: {abs(K_cap_est[1,1] - true_f):.2f} px')
print(f'cx error: {abs(K_cap_est[0,2] - true_c[0]):.2f} px')
print(f'cy error: {abs(K_cap_est[1,2] - true_c[1]):.2f} px')
print(f'\nTest reprojection RMSE: {np.sqrt((test_err**2).mean()):.3f} px')

**Capstone takeaways:**
- With low noise ($\sigma \le 1$ px), DLT recovers $K$ to within a few pixels.
- The reprojection error on the test set measures how well the estimated model generalises.
- In practice, multiple views of the calibration target improve robustness.

---

## Exercises

### Exercise 31.1: Triangle projection

Project a 3D triangle with vertices at $[0,0,3]$, $[1,0,3]$, $[0.5,1,3]$ using a pinhole
camera with $f = 400$, $c = (320, 240)$. Plot the projected triangle in the image plane.
Draw the edges connecting the vertices.

In [ ]:
# Your code here

### Exercise 31.2: FOV vs focal length

Write a function that computes the horizontal field of view for a given focal length $f_x$
and image width $w$. Plot FOV as a function of $f_x$ from 100 to 2000. At what focal length
is the FOV equal to 90 degrees?

In [ ]:
# Your code here

### Exercise 31.3: DLT calibration accuracy

Run the DLT calibration from Section 31.4 with pixel noise $\sigma$ varying from 0 to 5.
For each noise level, run 20 trials and record the mean absolute error in $f_x$.
Plot the error vs noise level. How does the error scale with $\sigma$?

In [ ]:
# Your code here

### Exercise 31.4: Undistortion

Given barrel distortion coefficients $k_1 = -0.3$, $k_2 = 0.05$, implement an iterative
undistortion algorithm. Start with the distorted point and iteratively refine the
undistorted coordinates. Verify that applying distortion to the undistorted result recovers
the original distorted point.

In [ ]:
# Your code here

### Exercise 31.5: Multi view calibration (challenge)

Generate a calibration grid viewed from 3 different camera poses. For each view, run DLT
to get three estimates of $K$. Average them. Compare the averaged $K$ to each individual
estimate. Does averaging improve accuracy?

In [ ]:
# Your code here